# 04. Deep Learning Forecasting Baselines: LSTM & Seq2Seq

## Background

The first wave of deep learning for time series centered on LSTMs and their seq2seq variants. While transformers have largely superseded them for long-horizon forecasting, LSTMs remain competitive for short-horizon, univariate forecasting and are far lighter to train and deploy.

**Seq2seq with teacher forcing**: the encoder processes historical context; the decoder autoregressively generates future predictions. Teacher forcing — using actual observations rather than predicted values during decoder training — stabilizes training but creates a train/test discrepancy (exposure bias).

**Temporal convolutional networks (TCNs)**: dilated causal convolutions expand receptive field exponentially with layers, matching LSTM performance with faster training. WaveNet (van den Oord 2016) pioneered this architecture.

## What You'll Learn

- LSTM encoder-decoder for multi-step forecasting
- Teacher forcing vs scheduled sampling
- Sequence normalization: RevIN (Reversible Instance Normalization)
- Comparing LSTM vs GBM on the same dataset

## Why This Matters

Understanding seq2seq architectures is prerequisite knowledge for modern forecasting transformers: TFT, PatchTST, and iTransformer all build on encoder-decoder concepts. The failure modes of LSTMs (vanishing gradients, inability to capture long-range dependencies) directly motivate transformer architectures.


## Dataset & Data Module

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Tuple

torch.manual_seed(42)
np.random.seed(42)

def generate_ts(n=600):
    t = np.arange(n)
    return 100 + 0.05*t + 10*np.sin(2*np.pi*t/52) + np.random.normal(0, 3, n)

raw = generate_ts(600)

class TimeSeriesDataset(Dataset):
    def __init__(self, series: np.ndarray, context_len: int, horizon: int):
        self.series = series
        self.context_len = context_len
        self.horizon = horizon

    def __len__(self):
        return len(self.series) - self.context_len - self.horizon + 1

    def __getitem__(self, idx):
        x = self.series[idx : idx + self.context_len]
        y = self.series[idx + self.context_len : idx + self.context_len + self.horizon]
        return torch.tensor(x, dtype=torch.float32).unsqueeze(-1), torch.tensor(y, dtype=torch.float32)

CONTEXT_LEN = 52
HORIZON = 12
SPLIT = int(len(raw) * 0.8)

# Normalize by training set statistics
mean, std = raw[:SPLIT].mean(), raw[:SPLIT].std()
normalized = (raw - mean) / std

train_ds = TimeSeriesDataset(normalized[:SPLIT], CONTEXT_LEN, HORIZON)
test_ds = TimeSeriesDataset(normalized[SPLIT:], CONTEXT_LEN, HORIZON)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(f'Dataset:')
print(f'  Series length: {len(raw)}')
print(f'  Context: {CONTEXT_LEN}, Horizon: {HORIZON}')
print(f'  Train batches: {len(train_loader)}')
print(f'  Test samples: {len(test_ds)}')


## LSTM Forecaster

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size: int = 1, hidden_size: int = 64,
                 num_layers: int = 2, horizon: int = 12, dropout: float = 0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Linear(hidden_size, horizon)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)          # (B, T, H)
        last = out[:, -1, :]            # (B, H) — use last hidden state
        return self.head(self.dropout(last))  # (B, horizon)

def train_model(model, train_loader, n_epochs=30, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(n_epochs):
        epoch_loss = 0
        for x, y in train_loader:
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}: loss={epoch_loss/len(train_loader):.4f}')
    return model

def evaluate_model(model, test_loader, mean, std):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x, y in test_loader:
            pred = model(x)
            all_preds.append(pred.numpy())
            all_targets.append(y.numpy())

    preds = np.concatenate(all_preds) * std + mean
    targets = np.concatenate(all_targets) * std + mean
    mape = np.mean(np.abs((targets - preds) / targets)) * 100
    rmse = np.sqrt(np.mean((targets - preds)**2))
    return mape, rmse

model = LSTMForecaster(hidden_size=64, num_layers=2, horizon=HORIZON)
params = sum(p.numel() for p in model.parameters())
print(f'LSTM model: {params:,} parameters')
print(f'\nTraining...')
model = train_model(model, train_loader, n_epochs=30)

mape, rmse = evaluate_model(model, test_loader, mean, std)
print(f'\nLSTM Forecaster Test Results:')
print(f'  MAPE: {mape:.2f}%')
print(f'  RMSE: {rmse:.4f}')
